In [ ]:
!pip install qiskit
!pip install qiskit-nature[pyscf]
!pip install qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 70.2 MB/s eta 0:00:00


In [ ]:
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
import numpy as np

In [ ]:
def uccsd(geometry, basis="sto-3g", max_qubits=12, match_hamiltonian_qubits=None):
    """
    Build UCCSD ansatz with active space reduction

    Args:
        geometry: Molecular geometry
        basis: Basis set
        max_qubits: Maximum qubits (ignored if match_hamiltonian_qubits is set)
        match_hamiltonian_qubits: Force ansatz to match specific qubit count

    Returns: (ansatz, initial_params, num_qubits)
    """

    # Convert geometry
    if isinstance(geometry, list):
        atom_string = "; ".join([f"{el} {c[0]:.4f} {c[1]:.4f} {c[2]:.4f}" for el, c in geometry])
    else:
        atom_string = geometry

    # Get molecular info
    driver = PySCFDriver(atom=atom_string, basis=basis, charge=0, spin=0, unit=DistanceUnit.ANGSTROM)
    problem_full = driver.run()
    if not isinstance(problem_full, ElectronicStructureProblem):
        problem_full = ElectronicStructureProblem(problem_full)

    # Determine active space
    full_electrons = sum(problem_full.num_particles)

    # Force specific qubit count if requested
    if match_hamiltonian_qubits is not None:
        target_qubits = match_hamiltonian_qubits
        print(f"Forcing UCCSD to match {target_qubits} qubits")
    else:
        target_qubits = min(max_qubits, max(6, 8))

    active_orbitals = target_qubits // 2

    # Ensure sufficient orbitals for electrons
    max_electrons_allowed = (active_orbitals - 1) * 2
    active_electrons = min(full_electrons, max_electrons_allowed)
    if active_electrons % 2 != 0:
        active_electrons -= 1

    electrons_per_spin = active_electrons // 2
    if active_orbitals <= electrons_per_spin:
        active_orbitals = electrons_per_spin + 2
        print(f"Warning: Adjusted to {active_orbitals} orbitals ({active_orbitals*2} qubits)")

    print(f"Active space: {active_electrons} electrons, {active_orbitals} orbitals → {active_orbitals*2} qubits")

    # Apply active space
    transformer = ActiveSpaceTransformer(
        num_electrons=active_electrons,
        num_spatial_orbitals=active_orbitals
    )
    problem_active = transformer.transform(problem_full)

    # Create ansatz
    mapper = JordanWignerMapper()
    ansatz = UCCSD(
        num_spatial_orbitals=problem_active.num_spin_orbitals // 2,
        num_particles=problem_active.num_particles,
        qubit_mapper=mapper
    )

    # Initial parameters (small random)
    if ansatz.num_parameters > 0:
        initial_params = np.random.normal(0, 0.02, ansatz.num_parameters)
    else:
        initial_params = np.array([])

    return ansatz, initial_params, ansatz.num_qubits


In [ ]:
def show_circuit(ansatz, max_gates=20):
    """Show circuit structure"""
    print(f"Circuit: {ansatz.num_qubits} qubits, {ansatz.num_parameters} params, depth {ansatz.depth()}")

    if len(ansatz.data) <= max_gates:
        print(ansatz)
    else:
        print(f"Circuit ({len(ansatz.data)} gates):")
        for i, inst in enumerate(ansatz.data[:max_gates]):
            qubits = [ansatz.find_bit(q).index for q in inst.qubits]
            print(f"  {inst.operation.name} {qubits}")
        if len(ansatz.data) > max_gates:
            print(f"  ... +{len(ansatz.data) - max_gates} more gates")

In [ ]:
def random_params(num_params, scale=0.02):
    """Generate small random parameters"""
    return np.random.normal(0, scale, num_params)

In [ ]:
def check_compatibility(hamiltonian, ansatz):
    """Check if Hamiltonian and ansatz have matching qubit counts"""
    h_qubits = hamiltonian.num_qubits
    a_qubits = ansatz.num_qubits

    if h_qubits != a_qubits:
        print(f"ERROR: Hamiltonian ({h_qubits} qubits) != Ansatz ({a_qubits} qubits)")
        print("VQE will fail - rebuild one of them to match")
        return False
    else:
        print(f"✓ Compatible: Both have {h_qubits} qubits")
        return True

In [ ]:
def uccsd_for_hamiltonian(geometry, hamiltonian, basis="sto-3g"):
    """Build UCCSD ansatz that matches existing Hamiltonian"""
    target_qubits = hamiltonian.num_qubits
    ansatz, params, qubits = uccsd(geometry, basis=basis, match_hamiltonian_qubits=target_qubits)

    if check_compatibility(hamiltonian, ansatz):
        print("✓ UCCSD ready for VQE")
        return ansatz, params
    else:
        raise ValueError("Could not create compatible UCCSD ansatz")

In [ ]:
def get_uccsd(geometry, max_qubits=12):
    """Get UCCSD ansatz and parameters"""
    return uccsd(geometry, max_qubits=max_qubits)


In [ ]:
def get_uccsd_for_hamiltonian(geometry, hamiltonian):
    """Get UCCSD that matches existing Hamiltonian"""
    return uccsd_for_hamiltonian(geometry, hamiltonian)

#give the hamiltonina as input (this is for getting parameter for n qubit if hamiltonian is n qubit)

In [ ]:
def quick_uccsd(geometry):
    """Ultra-simple UCCSD for small molecules"""
    ansatz, params, qubits = uccsd(geometry, max_qubits=8)
    print(f"UCCSD: {qubits} qubits, {len(params)} params")
    return ansatz, params


In [ ]:
if __name__ == "__main__":

    # NH3 geometry
    nh3 = [["N", [0,0,0]], ["H", [0.94,0,-0.38]], ["H", [-0.47,0.81,-0.38]], ["H", [-0.47,-0.81,-0.38]]]

    # Get UCCSD ansatz
    ansatz, params, qubits = uccsd(nh3, max_qubits=8)

    print(f"Parameters: {params}")
    print(f"Small random: {random_params(len(params))}")

    show_circuit(ansatz)

    # For VQE use:
    print(f"\nVQE ready:")
    print(f"ansatz = {ansatz}")
    print(f"initial_point = {params}")

Active space: 6 electrons, 4 orbitals → 8 qubits
Parameters: [-0.01366457  0.0195584  -0.00459375 -0.02103441 -0.00225608 -0.03383879
  0.02452011  0.0101996   0.02482076  0.00155922  0.02030728 -0.03855621
  0.02503358  0.00147743 -0.02684185]
Small random: [-2.25574484e-02  3.59333593e-02 -4.66976294e-05  1.86335836e-02
  3.13273670e-03  9.00963705e-03 -2.11764282e-02 -6.19872571e-03
  1.69983997e-02  1.13285354e-02 -7.65051834e-03  3.77383993e-02
 -1.60016509e-02 -6.57204036e-03 -2.76272496e-03]
Circuit: 8 qubits, 15 params, depth 1
     »
q_0: »
     »
q_1: »
     »
q_2: »
     »
q_3: »
     »
q_4: »
     »
q_5: »
     »
q_6: »
     »
q_7: »
     »
«     ┌──────────────────────────────────────────────────────────────────────────────────────────────┐
«q_0: ┤0                                                                                             ├
«     │                                                                                              │
«q_1: ┤1                      